# Python Programming

# Micro Project 

### DeadLine Reminder App

##### Emial setup

In [1]:
import ctypes
try:
    # Tell Windows to use native resolution for Tkinter
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

import tkinter as tk
from tkinter import ttk, messagebox
from tkcalendar import Calendar
from datetime import datetime
import pygame
import json
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- DEVELOPER CONFIGURATION ---
APP_SENDER_EMAIL = "deadlinerem@gmail.com" 
APP_SENDER_PASSWORD = "yygj qkij wlny byks"

DATA_FILE = "task_history.json"
CONFIG_FILE = "user_settings.json"

try:
    pygame.mixer.init()
except Exception:
    pass

pygame 2.6.1 (SDL 2.28.4, Python 3.10.8)
Hello from the pygame community. https://www.pygame.org/contribute.html


##### Fun def

In [2]:
def load_user_email():
    if os.path.exists(CONFIG_FILE):
        with open(CONFIG_FILE, "r") as f:
            return json.load(f).get("receiver_email", "")
    return ""

def save_user_email(email):
    with open(CONFIG_FILE, "w") as f:
        json.dump({"receiver_email": email}, f)

def load_tasks():
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, "r") as f:
            tasks = json.load(f)
            for task in tasks:
                task['datetime'] = datetime.strptime(task['datetime_str'], "%Y-%m-%d %H:%M")
                if task.get('completion_time_str'):
                    task['completion_time'] = datetime.strptime(task['completion_time_str'], "%Y-%m-%d %H:%M:%S")
                else:
                    task['completion_time'] = None
            return tasks
    return []

def save_tasks(tasks):
    export_tasks = []
    for t in tasks:
        task_copy = t.copy()
        task_copy['datetime_str'] = task_copy['datetime'].strftime("%Y-%m-%d %H:%M")
        del task_copy['datetime']
        
        if task_copy.get('completion_time'):
            task_copy['completion_time_str'] = task_copy['completion_time'].strftime("%Y-%m-%d %H:%M:%S")
        else:
            task_copy['completion_time_str'] = None
            
        if 'completion_time' in task_copy:
            del task_copy['completion_time']
            
        export_tasks.append(task_copy)
    with open(DATA_FILE, "w") as f:
        json.dump(export_tasks, f, indent=4)

##### alarm sound

In [3]:
def trigger_alarm(task_name):
    try:
        pygame.mixer.music.load("loud_alarm.mp3")
        pygame.mixer.music.play(-1)
    except:
        pass
    messagebox.showinfo("Deadline Reached!", f"Time is up for:\n\n{task_name}\n\nClick OK to stop the alarm.")
    try:
        pygame.mixer.music.stop()
    except:
        pass

def send_email_alert(task, receiver_email):
    if not APP_SENDER_EMAIL or not APP_SENDER_PASSWORD or not receiver_email:
        return 
    msg = MIMEMultipart()
    msg['From'] = APP_SENDER_EMAIL
    msg['To'] = receiver_email
    msg['Subject'] = f"DEADLINE ALERT: {task['name']}"
    
    time_string = task['datetime'].strftime('%Y-%m-%d %H:%M')
    body = f"Hello,\n\nYour deadline has been reached!\n\nTask: {task['name']}\nDescription: {task['desc']}\nTime: {time_string}\n\n- Deadline Reminder App"
    msg.attach(MIMEText(body, 'plain'))
    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(APP_SENDER_EMAIL, APP_SENDER_PASSWORD)
        server.send_message(msg)
        server.quit()
    except Exception as e:
        print(f"Failed to send email: {e}")

##### GUI

In [4]:
# Import Timepicker locally in this cell to prevent NameErrors in Jupyter
try:
    from tktimepicker import AnalogPicker, AnalogThemes, constants
    HAS_TIMEPICKER = True
except ImportError:
    HAS_TIMEPICKER = False

class DeadlineApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Deadline Reminder")
        self.root.geometry("1150x750")
        self.root.configure(bg="#f4f7f6")
        
        self.receiver_email = load_user_email()
        self.tasks = load_tasks()
        
        self.build_ui()
        self.refresh_ui()
        self.update_clock()
        self.check_time_loop()
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        
        if not self.receiver_email:
            self.root.after(500, self.open_settings)

    def build_ui(self):
        self.style = ttk.Style()
        self.style.theme_use("clam")
        self.style.configure("Treeview", font=("Segoe UI", 11), rowheight=28)
        self.style.configure("Treeview.Heading", font=("Segoe UI", 11, "bold"), background="#e0e6ed", foreground="#2b2d42")
        
        header = tk.Frame(self.root, bg="#2b2d42", pady=15)
        header.pack(fill=tk.X)
        
        tk.Label(header, text="Deadline Reminder", font=("Segoe UI", 20, "bold"), fg="white", bg="#2b2d42").pack(side=tk.LEFT, padx=20)
        self.clock_lbl = tk.Label(header, text="00:00:00", font=("Consolas", 18, "bold"), fg="#48c9b0", bg="#2b2d42")
        self.clock_lbl.pack(side=tk.LEFT, padx=30)
        
        ttk.Button(header, text="⚙️ Email Setup", command=self.open_settings).pack(side=tk.RIGHT, padx=20)

        dash_frame = tk.Frame(self.root, bg="#f4f7f6", pady=15)
        dash_frame.pack(fill=tk.X, padx=20)
        
        self.lbl_active = self.create_metric_card(dash_frame, "Active Deadlines", "#3498db")
        self.lbl_completed = self.create_metric_card(dash_frame, "Completed Tasks", "#2ecc71")
        self.lbl_missed = self.create_metric_card(dash_frame, "Missed Deadlines", "#e74c3c")

        content = tk.Frame(self.root, bg="#f4f7f6")
        content.pack(fill=tk.BOTH, expand=True, padx=20, pady=10)

        left_panel = tk.Frame(content, bg="#f4f7f6")
        left_panel.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 15))

        tk.Label(left_panel, text="⏳ Upcoming Tasks", font=("Segoe UI", 12, "bold"), bg="#f4f7f6").pack(anchor=tk.W)
        self.tree_up = self.create_treeview(left_panel)
        self.tree_up.pack(fill=tk.BOTH, expand=True, pady=(5, 15))
        self.tree_up.bind("<<TreeviewSelect>>", self.on_tree_up_select)
        
        tk.Label(left_panel, text="📁 History (Completed & Missed)", font=("Segoe UI", 12, "bold"), bg="#f4f7f6").pack(anchor=tk.W)
        self.tree_past = self.create_treeview(left_panel)
        self.tree_past.pack(fill=tk.BOTH, expand=True, pady=(5, 0))
        self.tree_past.bind("<<TreeviewSelect>>", self.on_tree_past_select)

        self.tree_past.tag_configure('ontime', foreground='#16a085') 
        self.tree_past.tag_configure('late', foreground='#d35400')   
        self.tree_past.tag_configure('missed', foreground='#c0392b') 
        self.tree_up.tag_configure('pending', foreground='#2980b9')  

        right_panel = tk.Frame(content, bg="#ffffff", bd=1, relief=tk.SOLID)
        right_panel.pack(side=tk.RIGHT, fill=tk.Y)
        right_panel.pack_propagate(False)
        right_panel.config(width=340)

        action_frame = tk.Frame(right_panel, bg="#ffffff", pady=15)
        action_frame.pack(fill=tk.X, padx=15)
        
        ttk.Button(action_frame, text="➕ New Task", command=self.open_add_window, width=20).pack(pady=5)
        ttk.Button(action_frame, text="✔️ Mark as Done", command=self.mark_done, width=20).pack(pady=5)
        ttk.Button(action_frame, text="🗑️ Delete Selected", command=self.delete_task, width=20).pack(pady=5)

        tk.Label(right_panel, text="Task Details:", font=("Segoe UI", 12, "bold"), bg="#ffffff").pack(anchor=tk.W, padx=15, pady=(15,0))
        self.desc_box = tk.Text(right_panel, font=("Segoe UI", 11), wrap=tk.WORD, bg="#f9fbfd", relief=tk.FLAT, padx=10, pady=10)
        self.desc_box.pack(fill=tk.BOTH, expand=True, padx=15, pady=15)

    def create_metric_card(self, parent, title, color):
        card = tk.Frame(parent, bg=color, width=200, height=80, pady=10)
        card.pack(side=tk.LEFT, padx=(0, 15), expand=True, fill=tk.BOTH)
        tk.Label(card, text=title, font=("Segoe UI", 11, "bold"), fg="white", bg=color).pack()
        lbl_val = tk.Label(card, text="0", font=("Segoe UI", 22, "bold"), fg="white", bg=color)
        lbl_val.pack()
        return lbl_val

    def create_treeview(self, parent):
        tree = ttk.Treeview(parent, columns=("Status", "Deadline", "Task"), show="headings", selectmode="browse", height=6)
        tree.heading("Status", text="Status")
        tree.heading("Deadline", text="Deadline")
        tree.heading("Task", text="Task Name")
        tree.column("Status", width=130, anchor=tk.CENTER)
        tree.column("Deadline", width=150, anchor=tk.CENTER)
        tree.column("Task", width=250)
        return tree

    def on_tree_up_select(self, event):
        for item in self.tree_past.selection():
            self.tree_past.selection_remove(item)
        self.show_details()

    def on_tree_past_select(self, event):
        for item in self.tree_up.selection():
            self.tree_up.selection_remove(item)
        self.show_details()

    def update_clock(self):
        current_time = datetime.now().strftime("%I:%M:%S %p")
        self.clock_lbl.config(text=current_time)
        self.root.after(1000, self.update_clock)

    def check_time_loop(self):
        now = datetime.now()
        updated = False
        
        for task in self.tasks:
            if task['status'] == 'Upcoming' and now >= task['datetime']:
                task['status'] = 'Missed' 
                updated = True
                
                if task.get('send_email'):
                    send_email_alert(task, self.receiver_email)
                if task.get('play_alarm'):
                    trigger_alarm(task['name'])
                    
        if updated:
            self.refresh_ui()
            save_tasks(self.tasks)
            
        self.root.after(1000, self.check_time_loop)

    def get_selected_task(self):
        sel_up = self.tree_up.selection()
        sel_past = self.tree_past.selection()
        
        if not sel_up and not sel_past:
            return None
            
        tree = self.tree_up if sel_up else self.tree_past
        sel_id = sel_up[0] if sel_up else sel_past[0]
        
        values = tree.item(sel_id, "values")
        
        # Safely extract values even if Tkinter truncates trailing empty columns
        dt_str = values[1]
        task_name = values[2] if len(values) > 2 else ""
        
        # Match by both Name and exactly formatted time to guarantee correct task
        for task in self.tasks:
            if task['name'] == task_name and task['datetime'].strftime("%Y-%m-%d %H:%M") == dt_str:
                return task
        return None

    def mark_done(self):
        task = self.get_selected_task()
        if not task:
            messagebox.showinfo("Selection Required", "Please select a task from the lists first.")
            return

        if task['status'] in ['Upcoming', 'Missed']:
            task['completion_time'] = datetime.now()
            
            if task['status'] == 'Upcoming':
                task['status'] = 'Completed On Time'
            elif task['status'] == 'Missed':
                task['status'] = 'Completed Late'
                
            save_tasks(self.tasks)
            self.refresh_ui()
            
            self.desc_box.config(state=tk.NORMAL)
            self.desc_box.delete("1.0", tk.END)
            self.desc_box.insert(tk.END, f"Successfully marked task as {task['status']}.\n\nPlease select a task to view full details.")
            self.desc_box.config(state=tk.DISABLED)
        else:
            messagebox.showinfo("Already Completed", "This task has already been marked as done.")

    def delete_task(self):
        task = self.get_selected_task()
        if task:
            if messagebox.askyesno("Confirm", f"Delete task?"):
                self.tasks.remove(task)
                save_tasks(self.tasks)
                self.refresh_ui()
                self.desc_box.config(state=tk.NORMAL)
                self.desc_box.delete("1.0", tk.END)
                self.desc_box.config(state=tk.DISABLED)

    def refresh_ui(self):
        for t in self.tree_up.get_children(): self.tree_up.delete(t)
        for t in self.tree_past.get_children(): self.tree_past.delete(t)
            
        active_cnt = comp_cnt = miss_cnt = 0
        sorted_tasks = sorted(self.tasks, key=lambda x: x['datetime'])
        
        for task in sorted_tasks:
            dt_str = task['datetime'].strftime("%Y-%m-%d %H:%M")
            if task['status'] == 'Upcoming':
                active_cnt += 1
                self.tree_up.insert("", tk.END, values=("⏳ Pending", dt_str, task['name']), tags=('pending',))
            else:
                if task['status'] == 'Completed On Time':
                    comp_cnt += 1
                    self.tree_past.insert("", tk.END, values=("🟢 On Time", dt_str, task['name']), tags=('ontime',))
                elif task['status'] == 'Completed Late':
                    comp_cnt += 1
                    self.tree_past.insert("", tk.END, values=("🟠 Done Late", dt_str, task['name']), tags=('late',))
                else:
                    miss_cnt += 1
                    self.tree_past.insert("", tk.END, values=("🔴 Missed", dt_str, task['name']), tags=('missed',))
                
        self.lbl_active.config(text=str(active_cnt))
        self.lbl_completed.config(text=str(comp_cnt))
        self.lbl_missed.config(text=str(miss_cnt))

    def show_details(self):
        task = self.get_selected_task()
        if task:
            self.desc_box.config(state=tk.NORMAL)
            self.desc_box.delete("1.0", tk.END)
            
            info = f"Task: {task['name']}\nDeadline: {task['datetime'].strftime('%B %d, %Y - %I:%M %p')}\nStatus: {task['status']}\n"
            if task.get('completion_time'):
                info += f"Finished At: {task['completion_time'].strftime('%B %d, %Y - %I:%M %p')}\n"
            info += f"\nDescription:\n{task['desc']}"
            
            self.desc_box.insert(tk.END, info)
            self.desc_box.config(state=tk.DISABLED)

    def open_settings(self):
        win = tk.Toplevel(self.root)
        win.title("Alert Preferences")
        win.geometry("350x200")
        win.grab_set()
        
        tk.Label(win, text="Welcome! Enter your email to receive alerts:", font=("Segoe UI", 10, "bold"), pady=15).pack()
        recv_entry = ttk.Entry(win, width=35)
        recv_entry.insert(0, self.receiver_email)
        recv_entry.pack(pady=5)

        def save():
            self.receiver_email = recv_entry.get().strip()
            save_user_email(self.receiver_email)
            win.destroy()

        ttk.Button(win, text="Save Email", command=save).pack(pady=20)

    def open_add_window(self):
        win = tk.Toplevel(self.root)
        win.title("New Task")
        win.geometry("450x800")
        win.grab_set()

        tk.Label(win, text="Task Name:", font=("Segoe UI", 10, "bold")).pack(pady=(15,2))
        name_entry = ttk.Entry(win, width=40)
        name_entry.pack()

        tk.Label(win, text="Description:", font=("Segoe UI", 10, "bold")).pack(pady=(15,2))
        desc_entry = tk.Text(win, width=40, height=4)
        desc_entry.pack()

        tk.Label(win, text="Select Date:", font=("Segoe UI", 10, "bold")).pack(pady=(15,2))
        cal = Calendar(win, selectmode='day', date_pattern='yyyy-mm-dd')
        cal.pack()

        tk.Label(win, text="Select Time:", font=("Segoe UI", 10, "bold")).pack(pady=(15,2))
        
        if HAS_TIMEPICKER:
            time_picker = AnalogPicker(win, type=constants.HOURS24)
            time_picker.pack(pady=5)
            theme = AnalogThemes(time_picker)
            theme.setDracula() 
        else:
            time_frame = tk.Frame(win)
            time_frame.pack()
            h_spin = ttk.Spinbox(time_frame, from_=0, to=23, width=5, format="%02.0f")
            h_spin.pack(side=tk.LEFT, padx=5)
            m_spin = ttk.Spinbox(time_frame, from_=0, to=59, width=5, format="%02.0f")
            m_spin.pack(side=tk.LEFT, padx=5)

        alarm_var, email_var = tk.BooleanVar(value=True), tk.BooleanVar(value=False)
        ttk.Checkbutton(win, text="Play Alarm", variable=alarm_var).pack(pady=(15,0))
        ttk.Checkbutton(win, text="Send Email", variable=email_var).pack(pady=5)

        def save():
            # Safeguard to prevent empty tasks from being created again
            entered_name = name_entry.get().strip()
            if not entered_name:
                messagebox.showwarning("Error", "Task Name cannot be empty!", parent=win)
                return
                
            try:
                if HAS_TIMEPICKER:
                    h, m = time_picker.time()
                else:
                    h, m = int(h_spin.get()), int(m_spin.get())
                    
                task_dt = datetime.strptime(f"{cal.get_date()} {h}:{m}", "%Y-%m-%d %H:%M")
                if task_dt <= datetime.now(): 
                    return messagebox.showwarning("Error", "Time must be in future!", parent=win)
                
                self.tasks.append({
                    'name': entered_name,
                    'desc': desc_entry.get("1.0", tk.END).strip(),
                    'datetime': task_dt,
                    'status': 'Upcoming',
                    'completion_time': None,
                    'play_alarm': alarm_var.get(),
                    'send_email': email_var.get()
                })
                save_tasks(self.tasks)
                self.refresh_ui()
                win.destroy()
            except Exception as e:
                messagebox.showerror("Error", f"Check your inputs. {e}", parent=win)

        ttk.Button(win, text="💾 Save Task", command=save, width=20).pack(pady=20)

    def on_closing(self):
        save_tasks(self.tasks)
        self.root.destroy()

##### APP

In [5]:
if __name__ == "__main__":
    root = tk.Tk()
    app = DeadlineApp(root)
    root.mainloop()